In [ ]:
%load_ext autoreload
%autoreload 2

import warnings, os, sys, time, regex

sys.path.append("/public1/biousers/jiachenli/PacBio/NDM.barcode")
import PacBioCCSsubprocess

import itertools

import pandas as pd
import numpy as np
import multiprocessing as mp

from plotnine import *

from dms_variants.codonvarianttable import CodonVariantTable
from dms_variants.constants import CBPALETTE
import dms_variants.illuminabarcodeparser
import dms_variants.plotnine_themes
from dms_variants.simulate import mutate_seq, rand_seq, simulate_CodonVariantTable
from dms_variants.utils import reverse_complement

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
warnings.simplefilter("ignore")

In [ ]:
current_dir = os.getcwd()
folder = "./output_files/kpc2_align_and_parse/"

variants_df = pd.read_csv(os.path.join(current_dir, folder)+ "/" + "NDM-1_PacBio_barcode-variant-table.csv", na_filter=None)

In [4]:
#patterns = r'(?e)(?P<p1>TCCTGAGTAGGACAAATCCG){e<=1}(?P<umi>[ATCG]{30})(?P<p2>GGATCCACTAGTTCTAGAAA){e<=1}'
patterns = regex.compile(
    r'(?e)(?P<p1>TAGTGGATCC){e<=1}'
    r'(?P<umi>[ATCG]{30})'
    r'(?P<p2>CGGATTTGTC){e<=1}'
)

barcode_list=variants_df["barcode"].tolist()
display(barcode_list)

['AAAAAAAAAGTCGACCGGATTTAATTAAGG',
 'AAAAAAAACAAGAAAGTCAGTTAATTAAGG',
 'AAAAAAAACACATTTATGATTTAATTAAGG',
 'AAAAAAAACTTACAGGCGCGTTAATTAAGG',
 'AAAAAAAATACATTTCGGATTTAATTAAGG',
 'AAAAAAAATGCAAACCAGCTTTAATTAAGG',
 'AAAAAAACAGATTATATAGGTTAATTAAGG',
 'AAAAAAAGATGGCACAGAACTTAATTAAGG',
 'AAAAAAAGATTATTCCCGTATTAATTAAGG',
 'AAAAAAAGGAGGAATGAGATTTAATTAAGG',
 'AAAAAAAGTTACAAAGACCATTAATTAAGG',
 'AAAAAAATGCCCGCGCCGGATTAATTAAGG',
 'AAAAAAATGTAGCAGGTGGTTTAATTAAGG',
 'AAAAAAATTATTAAGGTTACTTAATTAAGG',
 'AAAAAAATTCCGTTTCAATTTTAATTAAGG',
 'AAAAAAATTCTTTCGTCGGGTTAATTAAGG',
 'AAAAAACAATAGACTGTTCTTTAATTAAGG',
 'AAAAAACACACCACGAAGACTTAATTAAGG',
 'AAAAAACAGGTATGTGAGGTTTAATTAAGG',
 'AAAAAACAGTATTGTATGGATTAATTAAGG',
 'AAAAAACATTGAGTCGACTGTTAATTAAGG',
 'AAAAAACCATTTTTTATGTCTTAATTAAGG',
 'AAAAAACGAGACACAGCTCATTAATTAAGG',
 'AAAAAACGTAGTTAGAGGAATTAATTAAGG',
 'AAAAAACTAGGGGAGTGAGGTTAATTAAGG',
 'AAAAAACTGAACACAGAGAATTAATTAAGG',
 'AAAAAACTGAGGACTTTCATTTAATTAAGG',
 'AAAAAACTTGAGACGCTCCATTAATTAAGG',
 'AAAAAACTTGTGGTCTCT

In [ ]:
fq_folder="/public1/biousers/jiachenli/LJC_proj_kpc/kpcKP/rawdata"
fq_file=pd.read_csv(fq_folder + "/" + "rename.csv")
fq_name=fq_file["Sample"].tolist()
fq_list = list(a + b for a, b in itertools.product(fq_name, ["R1", "R2", "R3"]))
display(fq_list)

['kt640oR1',
 'kt640oR2',
 'kt640oR3',
 'nia1bkR1',
 'nia1bkR2',
 'nia1bkR3',
 'vhbjdyR1',
 'vhbjdyR2',
 'vhbjdyR3',
 'taskzuR1',
 'taskzuR2',
 'taskzuR3',
 'pct84qR1',
 'pct84qR2',
 'pct84qR3',
 'ly1r7gR1',
 'ly1r7gR2',
 'ly1r7gR3',
 't3rcfxR1',
 't3rcfxR2',
 't3rcfxR3',
 '5a9lbzR1',
 '5a9lbzR2',
 '5a9lbzR3']

In [20]:
print('Start to parallel process the files')
p = mp.Pool(len(fq_list))
for i in fq_list:
    res = p.apply_async(barcode_parser, args=(fq_folder + "/" + i + "_1.fq.gz", fq_folder + "/" + i + "_2.fq.gz", patterns, barcode_list, fq_folder + "/" + i + "_count"))

print('Waiting for all subprocesses done...')
p.close()
p.join()
time.sleep(20)
print('All subprocesses finish!')

Start to parallel process the files


Waiting for all subprocesses done...
All subprocesses finish!


In [ ]:
#bclen=20
#upstream="CTCTCCTGAGTAGGACAAATCCGCCTTAATTAA"
#downstream="GGATCCACTAGTTCTAGAAA"
#minq=20

#parser = dms_variants.illuminabarcodeparser.IlluminaBarcodeParser(
#    bclen=bclen,
#    valid_barcodes=variants_df["barcode"].tolist(),
#    upstream=upstream,
#    downstream=downstream,
#    upstream_mismatch=1,
#    downstream_mismatch=1,
#    minq=minq,
#)

#fq_folder="/public1/biousers/jiachenli/LJC_proj_kpc/kpcKP/rawdata"
#fq_file=pd.read_csv(fq_folder + "/" + "rename.csv")
#fq_name=fq_file["Sample"].tolist()
#fq_sample=fq_file["Abbr"].tolist()
#counts, fates = parser.parse(
#    r1files=fq_folder + "/" + fq_name[0] + "R1_1.fq.gz",
#    r2files=fq_folder + "/" + fq_name[0] + "R1_2.fq.gz",
#)